In [1]:
!git clone https://github.com/ducbao210/Video_deepfake_detection.git

Cloning into 'Video_deepfake_detection'...
remote: Enumerating objects: 593, done.
remote: Counting objects: 100% (593/593), done.
remote: Compressing objects: 100% (323/323), done.
remote: Total 593 (delta 351), reused 488 (delta 249), pack-reused 0 (from 0)
Receiving objects: 100% (593/593), 3.82 MiB | 16.11 MiB/s, done.
Resolving deltas: 100% (351/351), done.


In [2]:
%cd Video_deepfake_detection

/kaggle/working/Video_deepfake_detection


In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 99.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!pip install "hydra-core>=1.3.2" "omegaconf>=2.3.0" "ml_dtypes>=0.5.0"

In [5]:
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")

    with open('.env', 'w') as f:
        f.write(f"HF_TOKEN={hf_token}\n")
    print("Successfully created .env file.")

except Exception as e:
    print(f"Could not find HF_TOKEN in Secrets.\n{e}")

Successfully created .env file.


In [6]:
!mkdir -p data/processed
!gdown "https://drive.google.com/uc?id=1kqGpffyfES8MURpVTQcjjmwun92n7ABA" -O data/processed.zip
!unzip -q data/processed.zip -d data/
!rm data/processed.zip

Downloading...
From (original): https://drive.google.com/uc?id=1kqGpffyfES8MURpVTQcjjmwun92n7ABA
From (redirected): https://drive.google.com/uc?id=1kqGpffyfES8MURpVTQcjjmwun92n7ABA&confirm=t&uuid=0303bdf9-94ee-46ac-8334-b5c9ad472456
To: /kaggle/working/Video_deepfake_detection/data/processed.zip
100%|██████████████████████████████████████| 1.12G/1.12G [00:13<00:00, 84.3MB/s]


In [7]:
!python scripts/split_dataset.py

2026-08-08 14:42:28,239 | INFO | DEEPFAKEDETECTION | Found 3431 videos with extracted frames.
2026-08-08 14:42:28,241 | INFO | DEEPFAKEDETECTION | Found 28 unique actors: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28']
2026-08-08 14:42:28,241 | INFO | DEEPFAKEDETECTION | Train actors (16): ['01', '02', '03', '04', '05', '08', '09', '14', '16', '18', '20', '21', '23', '24', '26', '27']
2026-08-08 14:42:28,241 | INFO | DEEPFAKEDETECTION | Validation actors (6): ['06', '10', '12', '13', '15', '25']
2026-08-08 14:42:28,241 | INFO | DEEPFAKEDETECTION | Test actors (6): ['07', '11', '17', '19', '22', '28']
2026-08-08 14:42:28,276 | INFO | DEEPFAKEDETECTION | ========== SUMMARY ==========
2026-08-08 14:42:28,276 | INFO | DEEPFAKEDETECTION | STRATEGY: ACTOR_DISJOINT
2026-08-08 14:42:28,276 | INFO | DEEPFAKEDETECTION | TRAIN    | Total: 1215  | Real: 211  | Fake: 1004
2026-08

In [8]:
# Baseline - ConvNeXt
!python scripts/train.py model=convnext

2026-08-08 14:42:51,307 | INFO | CONVNEXT | Starting training for experiment: convnext
[*] Global random seed set to 42 for Python, NumPy, PyTorch, and cuDNN.
model.safetensors: 100%|█████████████████████| 201M/201M [00:02<00:00, 70.6MB/s]
2026-08-08 14:43:00,128 | INFO | CONVNEXT | MODEL: CONVNEXT
2026-08-08 14:43:00,128 | INFO | CONVNEXT | TOTAL PARAMETERS: 49,456,226
2026-08-08 14:43:00,129 | INFO | CONVNEXT | Training set class distribution: [211, 1004]
2026-08-08 14:43:00,129 | INFO | CONVNEXT | Class weights: [2.8791468143463135, 0.605079710483551]
2026-08-08 14:43:00,758 | INFO | CONVNEXT | No valid checkpoint found. Starting training from scratch.
2026-08-08 14:43:00,759 | INFO | CONVNEXT | Starting training for convnext on cuda...
2026-08-08 14:43:00,759 | INFO | CONVNEXT | 
[==================== Epoch 1/10 ====================]
Evaluating: 100%|██████████████████| 69/69 [00:26<00:00,  2.57it/s, loss=0.6503]
2026-08-08 14:44:50,009 | INFO | CONVNEXT | Train - ACCURACY: 0.6519 

In [9]:
# Baseline - ConvNeXt
!python scripts/train.py model=convnext

# Hybrid ConvNeXt-BiLSTM
!python scripts/train.py model=hybrid_bilstm

# Video Swin
!python scripts/train.py model=video_swin

# Knowledge Distillation - Teacher model: Video Swin - Student model: ConvNeXt
!python scripts/train_kd.py model=convnext_kd training=kd_training

# Timesformer
!python scripts/train.py model=timesformer

2026-08-08 15:02:38,674 | INFO | CONVNEXT | Starting training for experiment: convnext
[*] Global random seed set to 42 for Python, NumPy, PyTorch, and cuDNN.
2026-08-08 15:02:43,683 | INFO | CONVNEXT | MODEL: CONVNEXT
2026-08-08 15:02:43,683 | INFO | CONVNEXT | TOTAL PARAMETERS: 49,456,226
2026-08-08 15:02:43,684 | INFO | CONVNEXT | Training set class distribution: [211, 1004]
2026-08-08 15:02:43,684 | INFO | CONVNEXT | Class weights: [2.8791468143463135, 0.605079710483551]
2026-08-08 15:02:44,272 | INFO | CONVNEXT | No valid checkpoint found. Starting training from scratch.
2026-08-08 15:02:44,272 | INFO | CONVNEXT | Starting training for convnext on cuda...
2026-08-08 15:02:44,273 | INFO | CONVNEXT | 
[==================== Epoch 1/10 ====================]
Evaluating: 100%|██████████████████| 69/69 [00:28<00:00,  2.45it/s, loss=0.6503]
2026-08-08 15:04:34,673 | INFO | CONVNEXT | Train - ACCURACY: 0.6519 - BALANCED_ACCURACY: 0.4899 - PRECISION: 0.8224 - RECALL: 0.7380 - F1: 0.7780 - A

In [10]:
# Evaluate ConvNeXt
!python scripts/evaluate.py model=convnext inference.checkpoint=outputs/convnext/checkpoints/best.pth

# Evaluate Hybrid BiLSTM
!python scripts/evaluate.py model=hybrid_bilstm inference.checkpoint=outputs/hybrid_bilstm/checkpoints/best.pth

# Evaluate Video Swin
!python scripts/evaluate.py model=video_swin inference.checkpoint=outputs/video_swin/checkpoints/best.pth

# Evaluate model Student - backbone is still convnext 
!python scripts/evaluate.py model=convnext inference.checkpoint=outputs/convnext_kd/checkpoints/best.pth


# Evaluate Timeformer
!python scripts/evaluate.py model=timesformer inference.checkpoint=outputs/timesformer/checkpoints/best.pth

[*] Global random seed set to 42 for Python, NumPy, PyTorch, and cuDNN.
2026-08-08 17:13:06,449 | INFO | CONVNEXT | MODEL: CONVNEXT
2026-08-08 17:13:06,449 | INFO | CONVNEXT | Loading checkpoint from: outputs/convnext/checkpoints/best.pth
2026-08-08 17:13:06,450 | INFO | CONVNEXT | Checkpoint not found locally at outputs/convnext/checkpoints/best.pth. Attempting to download from Hugging Face...
2026-08-08 17:13:06,451 | INFO | CONVNEXT | Downloading checkpoints/convnext/best.pth from repo ducbao210/video-deepfake-detection...
checkpoints/convnext/best.pth: 100%|█████████| 322M/322M [00:05<00:00, 61.5MB/s]
2026-08-08 17:13:12,099 | INFO | CONVNEXT | Download complete! Checkpoint saved at: outputs/convnext/checkpoints/best.pth
Predicting: 100%|███████████████████████████████| 69/69 [00:28<00:00,  2.38it/s]
2026-08-08 17:13:41,547 | INFO | CONVNEXT | Optimal F1 threshold on the validation set: 0.25
Predicting: 100%|███████████████████████████████| 24/24 [00:10<00:00,  2.25it/s]
2026-08-08

In [11]:
import shutil
from pathlib import Path

processed_dir = Path("data/processed")
if processed_dir.exists():
    shutil.rmtree(processed_dir)
    print("Deleted:", processed_dir)

split_dir = Path("data/split")
if split_dir.exists():
    shutil.rmtree(split_dir)
    print("Deleted:", split_dir)

shutil.make_archive(
    "/kaggle/working/deepfake_outputs",
    "zip",
    "outputs",
)

print("Saved:", "/kaggle/working/deepfake_outputs.zip")

Deleted: data/processed
Deleted: data/split
Saved: /kaggle/working/deepfake_outputs.zip
